# Deterministic Performance

In [ ]:
SEED = 69
ENABLE_DETERMINISM = True

import os
os.environ['PYTHONHASHSEED'] = f'{SEED}'
os.environ['TF_DETERMINISTIC_OPS'] = f'{ENABLE_DETERMINISM}'
os.environ['TF_CUDNN_DETERMINISTIC'] = f'{ENABLE_DETERMINISM}'

import random
import numpy as np
import tensorflow as tf
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Downloading dataset from Roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="7cJr2ptqK1B8pDl1nHHS")
project = rf.workspace("meow-vhmhx").project("main-dataset-6b2cv")
version = project.version(3)
dataset = version.download("folder")

# Classes Counts

In [ ]:
base_path = "/content/Main-DataSet-3"  # change to your dataset path
splits = ["train", "valid", "test"]

# Dictionary to store total images per class
total_dataset = {}

for split in splits:
    split_path = os.path.join(base_path, split)
    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
            if class_name in total_dataset:
                total_dataset[class_name] += count
            else:
                total_dataset[class_name] = count

# total per class
print("=== Total Dataset ===")
for class_name, count in total_dataset.items():
    print(f"{class_name}: {count} images")

# total number of images across all classes
total_images = sum(total_dataset.values())
print(f"\nTOTAL IMAGES IN DATASET: {total_images}")


# Splitting dataset

In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_dir = "/content/Main-DataSet-3/train"
val_dir = "/content/Main-DataSet-3/valid"
test_dir = "/content/Main-DataSet-3/test"

img_size = (224, 224)
batch_size = 32

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
)

test_ds = image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
)

# # DETERMINISM
# options = tf.data.Options()
# options.experimental_deterministic = ENABLE_DETERMINISM

# train_ds = train_ds.with_options(options)
# val_ds   = val_ds.with_options(options)
# test_ds  = test_ds.with_options(options)

# Model creation

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

pretrained_model = MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
## all layers trainable
pretrained_model.trainable = True
## all layers except batchnorm trainable
# pretrained_model.trainable = False
# for layer in pretrained_model.layers:
#     if not isinstance(layer, tf.keras.layers.BatchNormalization):
#         layer.trainable = True
## last 40 layers trainable
# pretrained_model.trainable = False
# for layer in pretrained_model.layers[-40:]:
#     layer.trainable = True

inputs = Input(shape=(224, 224, 3))
x = pretrained_model(inputs)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(6, activation='softmax')(x)
model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Training

In [ ]:
callbacks = [
    # ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=2, min_lr=5e-7),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    batch_size=32
)

# Saving & Loading model

In [ ]:
print(model.evaluate(test_ds))
model.save("model.keras")

model_loaded = load_model("/content/model.keras")
print(model_loaded.evaluate(test_ds))

In [ ]:
from google.colab import files
files.download("/content/model.keras")

# Metrics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def model_evaluate(model, model_name):
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)

    print(f"\n{model_name}:")
    print(f"\tTrain acc: {train_acc*100:>6.2f}%, loss: {train_loss:>6.4f}")
    print(f"\tVal   acc: {  val_acc*100:>6.2f}%, loss: {  val_loss:>6.4f}")
    print(f"\tTest  acc: { test_acc*100:>6.2f}%, loss: { test_loss:>6.4f}")

def plot_cm(model, model_name):
    _, axes = plt.subplots(1, 3, figsize=(18, 5))

    train_true = np.concatenate([labels.numpy() for _, labels in train_ds], axis=0)
    train_pred = model.predict(train_ds, verbose=0)
    train_pred = np.argmax(train_pred, axis=1)
    train_cm = confusion_matrix(train_true, train_pred)
    train_disp = ConfusionMatrixDisplay(confusion_matrix=train_cm, display_labels=train_ds.class_names)
    train_disp.plot(ax=axes[0], cmap="Blues")
    axes[0].set_title(f"{model_name} Training")

    val_true = np.concatenate([labels.numpy() for _, labels in val_ds], axis=0)
    val_pred = model.predict(val_ds, verbose=0)
    val_pred = np.argmax(val_pred, axis=1)
    val_cm = confusion_matrix(val_true, val_pred)
    val_disp = ConfusionMatrixDisplay(confusion_matrix=val_cm, display_labels=val_ds.class_names)
    val_disp.plot(ax=axes[1], cmap="Blues")
    axes[1].set_title(f"{model_name} Validation")

    test_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
    test_pred = model.predict(test_ds, verbose=0)
    test_pred = np.argmax(test_pred, axis=1)
    test_cm = confusion_matrix(test_true, test_pred)
    test_disp = ConfusionMatrixDisplay(confusion_matrix=test_cm, display_labels=test_ds.class_names)
    test_disp.plot(ax=axes[2], cmap="Blues")
    axes[2].set_title(f"{model_name} Testing")

    plt.tight_layout()
    plt.show()

In [ ]:
model_evaluate(model_loaded, "model")
plot_cm(model_loaded, "model")

# Manual Testing

In [ ]:
import requests
from io import BytesIO
from PIL import Image, UnidentifiedImageError

# Predict multiple images in one run
def load_and_prepare(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img = img.resize((224, 224))
        img_array = tf.keras.utils.img_to_array(img)
        return img_array
    except (requests.exceptions.RequestException, UnidentifiedImageError, ValueError) as e:
        print(f"Could not load or process image from {url}: {e}")
        return None

def predict_batch(model, image_urls):
    images = []
    valid_urls_indices = [] # Keep track of original indices for valid images

    for i, url in enumerate(image_urls):
        img_array = load_and_prepare(url)
        if img_array is not None:
            images.append(img_array)
            valid_urls_indices.append(i)

    if not images:
        print("No valid images to predict.")
        return [], []

    # Convert to batch
    batch = np.array(images)
    predictions = model.predict(batch)
    return predictions, valid_urls_indices

# Example usage:
image_urls = [
    "https://images.stockcake.com/public/2/4/c/24c6e859-f3c2-464a-b77c-2793c60e3d0d_large/snowy-residential-street-stockcake.jpg",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS5c1gR5o3f3KT3x84_VsLtbKwNdDJ9Z3oPDw&s",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTKNC_Q6YxA4kz6tbP-nV7uEw2Pswui3W6UIA&s",
    "https://www.shutterstock.com/image-photo/snow-covered-city-street-during-260nw-2118338555.jpg",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQY6Szr55cbX_8yN9e-9svWVSw61VqxVdW04Q&s",
    "https://images.stockcake.com/public/8/3/c/83c9f72a-5ec6-41d6-9a0e-dca646065ecc_large/snowy-urban-street-stockcake.jpg",
    "https://static01.nyt.com/images/2018/02/27/world/27Rome2/27Rome2-articleLarge.jpg?quality=75&auto=webp&disable=upscale",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQKrc67Y_jg3xpXk8g_dfrLohYckPm3aQwTcQ&s",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSysuxhbY41Vfv780-mbdnLrxANvArudq1UuQ&s",
    "https://images.pexels.com/photos/31460661/pexels-photo-31460661.jpeg",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQTk6bxJh1gxZyeA0FvS-cG43MYMSa74Tvsig&s",
    "https://images.pexels.com/photos/15819943/pexels-photo-15819943.jpeg",
    "https://images.pexels.com/photos/15819943/pexels-photo-15819943.jpeg",
    "https://i0.wp.com/yaleclimateconnections.org/wp-content/uploads/2025/02/0225_GettyImages-2198978243_1600px.jpg?fit=1600%2C900&ssl=1",
    "https://preview.redd.it/chicago-2025-11-29-v0-wth14ih0ha4g1.jpeg?width=640&crop=smart&auto=webp&s=811758fa61a672e049a47e7523e423448c3768a3"
]

preds, valid_indices = predict_batch(model, image_urls)

class_names = train_ds.class_names
# class_names = ["Cloudy","Snowy","foggy","night","rainy","sunny"]

if preds is not None and len(preds) > 0:
    for i, p in enumerate(preds):
        original_url_index = valid_indices[i]
        label = class_names[p.argmax()]
        # print(f"Image {original_url_index+1} (from {image_urls[original_url_index]}): {label}")
        print(f"Image {original_url_index+1}: {label}")